In [2]:
from xgboost import XGBClassifier

In [3]:
import pandas as pd
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

URL = "https://archive.ics.uci.edu/static/public/" "222/bank+marketing.zip"
outer = ZipFile(BytesIO(urlopen(URL).read()))
inner = ZipFile(BytesIO(outer.read("bank-additional.zip")))
df = pd.read_csv(inner.open("bank-additional/bank-additional.csv"), sep=";")
print(df.shape)

(4119, 21)


In [4]:
model = XGBClassifier(n_estimators=200, learning_rate=0.05, random_state=42)

In [5]:
X = df.drop(columns=["duration", "y"])
y = df["y"]

X_encoded = pd.get_dummies(X, drop_first=True)
y_encoded = df["y"].map({"yes": 1, "no": 0})


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, stratify=y_encoded
)

In [6]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)


def metrics(X_test, y_test, model):
    y_pred = model.predict(X_test)
    y_probs = model.predict_proba(X_test)[:, 1]

    result = {
        "Accuracy": round(accuracy_score(y_test, y_pred), 2),
        "Precision": round(precision_score(y_test, y_pred), 2),
        "Recall": round(recall_score(y_test, y_pred), 2),
        "F1 Score": round(f1_score(y_test, y_pred), 2),
        "ROC AUC": round(roc_auc_score(y_test, y_probs), 2),
    }

    return pd.DataFrame(result, index=[0])

In [7]:
model.fit(X_train, y_train)

metrics(X_test, y_test, model)

# 	Accuracy	Precision	Recall	F1 Score	ROC AUC
# 0	  0.9	     0.595	   0.278	 0.379	   0.753

,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.65,0.27,0.38,0.8


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [50, 70, 100, 200, 500],
    "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
best_model = grid_search.best_estimator_

metrics(X_test, y_test, grid_search)

Best Parameters: {'learning_rate': 0.01, 'n_estimators': 100}
Best CV Score: 0.8974203338391502


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.91,0.11,0.2,0.82


In [14]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    # "n_estimators": [50, 70, 100, 200, 500],
    # "learning_rate": [0.01, 0.05, 0.1, 0.2, 0.5],
    "max_depth": [2, 4, 8, 9, 10],
    "min_child_weight": [1, 2, 3, 4, 5],
    "gamma": [0, 1],
}

grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=5,
    scoring="precision",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best CV Score:", grid_search.best_score_)
best_model = grid_search.best_estimator_

print(model.score(X_train, y_train))

metrics(X_test, y_test, grid_search)

Best Parameters: {'gamma': 1, 'max_depth': 2, 'min_child_weight': 3}
Best CV Score: 0.6931607865818392
0.9453717754172989


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.9,0.71,0.19,0.3,0.81


In [ ]:
model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=4,
    min_child_weight=1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

print(model.score(X_train, y_train))
metrics(X_test, y_test, model)

0.9277693474962063


,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,0.91,0.7,0.26,0.37,0.83
